# SML 301

## Session 13: Recurrence

* RNN
* LSTM
* GRU

We will work through the [NLP From Scratch: Classifying Names with a Character-Level RNN](https://docs.pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial.html) from PyTorch by Sean Robertson

In [ ]:
# install packages (if need be)


In [ ]:
#import needed libraries
import glob
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os
import random
import string
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import unicodedata

from io import open
from torch.utils.data import Dataset, Subset

In [ ]:
# Check if CUDA is available
device = torch.device('cpu')
if torch.cuda.is_available():
    device = torch.device('cuda')

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

In [ ]:
# We can use "_" to represent an out-of-vocabulary character, that is, any character we are not handling in our model
allowed_characters = string.ascii_letters + " .,;'" + "_"
n_letters = len(allowed_characters)

# Turn a Unicode string to plain ASCII, thanks to https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in allowed_characters
    )

# example
print (f"converting 'Ślusàrski' to {unicodeToAscii('Ślusàrski')}")

In [ ]:
# Find letter index from all_letters, e.g. "a" = 0
def letterToIndex(letter):
    # return our out-of-vocabulary character if we encounter a letter unknown to our model
    if letter not in allowed_characters:
        return allowed_characters.find("_")
    else:
        return allowed_characters.find(letter)

# Turn a line into a <line_length x 1 x n_letters>,
# or an array of one-hot letter vectors
def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_letters)
    for li, letter in enumerate(line):
        tensor[li][0][letterToIndex(letter)] = 1
    return tensor

# examples
print (f"The letter 'a' becomes \n {lineToTensor('a')}") #notice that the first position in the tensor = 1
print (f"The letter 'A' becomes \n {lineToTensor('A')}") #notice that the first position in the tensor = 27
#print (f"The name 'Derek' becomes \n {lineToTensor('Derek')}")

## Load Data

Included in the `data/names` directory are 18 text files named as [Language].txt. Each file contains a bunch of names, one name per line, mostly romanized (but we still need to convert from Unicode to ASCII).

In [ ]:
class NamesDataset(Dataset):

    def __init__(self, data_dir):
        self.data_dir = data_dir #for provenance of the dataset
        self.load_time = time.localtime #for provenance of the dataset
        labels_set = set() #set of all classes

        self.data = []
        self.data_tensors = []
        self.labels = []
        self.labels_tensors = []

        #read all the ``.txt`` files in the specified directory
        text_files = glob.glob(os.path.join(data_dir, '*.txt'))
        for filename in text_files:
            label = os.path.splitext(os.path.basename(filename))[0]
            labels_set.add(label)
            lines = open(filename, encoding='utf-8').read().strip().split('\n')
            for name in lines:
                self.data.append(name)
                self.data_tensors.append(lineToTensor(name))
                self.labels.append(label)

        #Cache the tensor representation of the labels
        self.labels_uniq = list(labels_set)
        for idx in range(len(self.labels)):
            temp_tensor = torch.tensor([self.labels_uniq.index(self.labels[idx])], dtype=torch.long)
            self.labels_tensors.append(temp_tensor)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_item = self.data[idx]
        data_label = self.labels[idx]
        data_tensor = self.data_tensors[idx]
        label_tensor = self.labels_tensors[idx]

        return label_tensor, data_tensor, data_label, data_item

def label_from_output(output, output_labels):
    top_n, top_i = output.topk(1)
    label_i = top_i[0].item()
    return output_labels[label_i], label_i

#alldata = NamesDataset("data/names")
#alldata = NamesDataset("")
#alldata = NamesDataset("https://github.com/dsollberger/sml301slides/blob/main/data/names")
#print(f"loaded {len(alldata)} items of data")
#print(f"example = {alldata[0]}")

In [ ]:
torch.save(alldata, 'alldata.pt')

In [ ]:
alldata = torch.load('alldata.pt', weights_only = False)

In [ ]:
# subset (only to ease exploratory calculations)
#alldata = torch.utils.data.Subset(alldata, list(range(0 , len(alldata), 10)))

train_set, test_set = torch.utils.data.random_split(alldata, [.10, .90], generator=torch.Generator(device=device).manual_seed(301))

print(f"train examples = {len(train_set)}, validation examples = {len(test_set)}")

# RNN

Recurrent Neural Networks

## Create Model

In [ ]:
# define model
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharRNN, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_size) #RNN layer
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, line_tensor):
        rnn_out, hidden = self.rnn(line_tensor)
        output = self.h2o(hidden[0])
        output = self.softmax(output)

        return output

In [ ]:
n_hidden = 64 #HYPERPARAMETER: hidden layer size
rnn = CharRNN(n_letters, n_hidden, len(alldata.labels_uniq))
print(rnn)

In [ ]:
# make prediction (before training??)
input = lineToTensor('Derek')
output = rnn(input) #this is equivalent to ``output = rnn.forward(input)``
print(output)
print(label_from_output(output, alldata.labels_uniq))

## Train Model

In [ ]:
# train model
def train(rnn, training_data, n_epoch = 10, n_batch_size = 64, report_every = 50, learning_rate = 0.2, criterion = nn.NLLLoss()):
    """
    Learn on a batch of training_data for a specified number of iterations and reporting thresholds
    """
    # Keep track of losses for plotting
    current_loss = 0
    all_losses = []
    rnn.train()
    optimizer = torch.optim.SGD(rnn.parameters(), lr=learning_rate)

    start = time.time()
    print(f"training on data set with n = {len(training_data)}")

    for iter in range(1, n_epoch + 1):
        rnn.zero_grad() # clear the gradients

        # create some minibatches
        # we cannot use dataloaders because each of our names is a different length
        batches = list(range(len(training_data)))
        random.shuffle(batches)
        batches = np.array_split(batches, len(batches) //n_batch_size )

        for idx, batch in enumerate(batches):
            batch_loss = 0
            for i in batch: #for each example in this batch
                (label_tensor, text_tensor, label, text) = training_data[i]
                output = rnn.forward(text_tensor)
                loss = criterion(output, label_tensor)
                batch_loss += loss

            # optimize parameters
            batch_loss.backward()
            nn.utils.clip_grad_norm_(rnn.parameters(), 3)
            optimizer.step()
            optimizer.zero_grad()

            current_loss += batch_loss.item() / len(batch)

        all_losses.append(current_loss / len(batches) )
        if iter % report_every == 0:
            print(f"{iter} ({iter / n_epoch:.0%}): \t average batch loss = {all_losses[-1]}")
        current_loss = 0

    return all_losses

In [ ]:
start = time.time()
all_losses = train(rnn, train_set, n_epoch = 15, learning_rate = 0.301, report_every = 3)
end = time.time()
print(f"training took {end-start}s")

In [ ]:
# make prediction (after training)
input = lineToTensor('Derek')
output = rnn(input)
print(output)
print(label_from_output(output, alldata.labels_uniq))

## Evaluate Model

In [ ]:
def evaluate(rnn, testing_data, classes):
    confusion = torch.zeros(len(classes), len(classes))

    rnn.eval() #set to eval mode
    with torch.no_grad(): # do not record the gradients during eval phase
        for i in range(len(testing_data)):
            (label_tensor, text_tensor, label, text) = testing_data[i]
            output = rnn(text_tensor)
            guess, guess_i = label_from_output(output, classes)
            label_i = classes.index(label)
            confusion[label_i][guess_i] += 1

    # Normalize by dividing every row by its sum
    for i in range(len(classes)):
        denom = confusion[i].sum()
        if denom > 0:
            confusion[i] = confusion[i] / denom

    # Set up plot
    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(confusion.cpu().numpy()) #numpy uses cpu here so we need to use a cpu version
    fig.colorbar(cax)

    # Set up axes
    ax.set_xticks(np.arange(len(classes)), labels=classes, rotation=90)
    ax.set_yticks(np.arange(len(classes)), labels=classes)

    # Force label at every tick
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    # sphinx_gallery_thumbnail_number = 2
    plt.show()

In [ ]:
plt.figure()
plt.plot(all_losses)
plt.show()

In [ ]:
evaluate(rnn, test_set, classes=alldata.labels_uniq)

# LSTM

Long Short Term Memory

## Create Model

In [ ]:
# define model
class CharLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharLSTM, self).__init__()

        self.lstm = nn.LSTM(input_size, hidden_size) #LSTM layer
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, line_tensor):
        lstm_out, hidden = self.lstm(line_tensor)
        output = self.h2o(hidden[0])
        output = self.softmax(output)

        return output

In [ ]:
n_hidden = 64 #HYPERPARAMETER: hidden layer size
lstm = CharLSTM(n_letters, n_hidden, len(alldata.labels_uniq))
print(lstm)

## Train Model

In [ ]:
start = time.time()
all_losses = train(lstm, train_set, n_epoch = 15, learning_rate = 0.301, report_every = 3)
end = time.time()
print(f"training took {end-start}s")

In [ ]:
# make prediction (after training)
input = lineToTensor('Derek')
output = lstm(input)
print(output)
print(label_from_output(output, alldata.labels_uniq))

## Evaluate Model

In [ ]:
plt.figure()
plt.plot(all_losses)
plt.show()

In [ ]:
evaluate(lstm, test_set, classes=alldata.labels_uniq)

# GRU

Gated Recurrence Unit

## Create Model


In [ ]:
# define model
class CharGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharGRU, self).__init__()

        self.gru = nn.GRU(input_size, hidden_size) #GRU layer
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, line_tensor):
        gru_out, hidden = self.gru(line_tensor)
        output = self.h2o(hidden[0])
        output = self.softmax(output)

        return output

In [ ]:
n_hidden = 64 #HYPERPARAMETER: hidden layer size
gru = CharGRU(n_letters, n_hidden, len(alldata.labels_uniq))
print(gru)

## Train Model

In [ ]:
start = time.time()
all_losses = train(gru, train_set, n_epoch = 15, learning_rate = 0.301, report_every = 3)
end = time.time()
print(f"training took {end-start}s")

In [ ]:
# make prediction (after training)
input = lineToTensor('Derek')
output = gru(input)
print(output)
print(label_from_output(output, alldata.labels_uniq))

## Evaluate Model

In [ ]:
plt.figure()
plt.plot(all_losses)
plt.show()

In [ ]:
evaluate(gru, test_set, classes=alldata.labels_uniq)